In [246]:
import numpy as np
import random
import time
from IPython.display import display, clear_output

# https://www.learndatasci.com/tutorials/reinforcement-q-learning-scratch-python-openai-gym/

In [247]:
#
#       -----------    state: 
#      |   start   |     0      (0 reward for being in this state)
#      |     -1    |     1
#      |     +2    |     2
#      |     -1    |     3
#      |     -2    |     4
#      |     -3    |     5
#      |     -4    |     6
#      |     +10   |     7
#      -------------

In [248]:
class env_class:
    def __init__(self):
        self.s = 0       
        self.action_space = ['up','down','stay']

    def render(self):   # print current state of environment  
        statelist = [ str(x).zfill(2) for x in range(0,8) ]
        statelist[self.s] = statelist[self.s] + ' *'  # draw in a * where the player is
        print(*statelist, sep='\n')
        print('\n')
            
    def reset(self):    # reset state back to 0
        self.s = 0
        
    def step(self, action):
        if action=='down' and self.s < 7:
            self.s += 1
        elif action=='up' and self.s > 0:
            self.s -= 1
        else:
            pass
        
    def reward_from_action(self, action):
        if action=='down' and self.s < 7:
            next_state = self.s +1
        elif action=='up' and self.s > 0:
            next_state = self.s -1
        else:
            next_state = self.s
        return [0,-1,2,-1,-2,-3,-4,10][next_state]
            
env = env_class()

In [249]:
# the current state of the environment is stored in "s":
env.s

0

In [250]:
# the space of possible actions is stored in "action_space":
env.action_space

['up', 'down', 'stay']

In [251]:
# taking an action updates the state:
for action_i in ['down','down','up','down','stay','down','down','stay','up']:
    print(f'{[action_i]}')
    env.step(action_i)
    print(f'state: {env.s}\n')
    env.render()
    clear_output(wait=True)
    time.sleep(1)

['up']
state: 3

00
01
02
03 *
04
05
06
07




In [252]:
# env.reward_from_action() gives the reward for a given action, 
# given the state that we're currently in
print( f'current state: {env.s} ' )
print( f'reward if move up: {env.reward_from_action("up")}' )
print( f'reward if stay: {env.reward_from_action("stay")}' )
print( f'reward if move down: {env.reward_from_action("down")}' )

current state: 3 
reward if move up: 2
reward if stay: -1
reward if move down: -2


In [253]:
# the state can be reset back to the starting state again using env.reset():
env.reset()
env.render()

00 *
01
02
03
04
05
06
07




In [279]:
q_table = np.zeros([8, 3])
q_table

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

In [280]:
# Hyperparameters
alpha = 0.3             # 
gamma = 0.8             # discount rate
epsilon = 1           # probability of exploration

# length of each game (episode)
episode_length = 100

In [283]:
env.reset()
step_counter = 0    # to keep track of when each game (episode) has ended
episode_counter = 1

for i in range(1000):
    print(f'episode {episode_counter}')
    print(f'step {step_counter}\n')
    print(f'epsilon = {epsilon}')
    
    # reduce epsilon:
    epsilon = max( 0.05, epsilon*0.9999 )
    
    # choose an action:    
    if random.uniform(0, 1) < epsilon:
        print('explore')
        action = random.sample( env.action_space, 1 )[0]  # explore
    else:
        print('exploit')
        action = env.action_space[ np.argmax( q_table[env.s] ) ]  # exploit
    
    print(f'[{action}]')
    
    print(f'state {env.s}')
    env.render()    
        
    # get reward for action:
    reward = env.reward_from_action( action )    
    
    action_index = np.where( [ x==action for x in env.action_space ] )[0][0]
    print( action_index )
    state = env.s
    old_value = q_table[state, action_index]
    
    # update state:
    env.step( action )
    next_max = np.max( q_table[env.s] )     # max for next state 
    
    # update Q-table:
    new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
    q_table[state, action_index] = new_value
    print( env.action_space )
    print( q_table )                  
    
    # iterate step counter:
    step_counter += 1
    
    if step_counter >= episode_length:
        env.reset()
        episode_counter += 1
        step_counter = 0
    else:    
        step_counter += 1    
    
    clear_output(wait=True)
    time.sleep(0.05)
    

KeyboardInterrupt: 